###Модуль 3: Enterprise LangChain. Агенты, Инструменты и LCEL.
**Цель модуля:**
Изучить архитектуру масштабируемых систем на базе чистого **LangChain (LCEL)**. Мы разберем, как управлять инструментами через Pydantic-схемы, как строить семантические роутеры для экономии ресурсов, как динамически подгружать инструменты (Tool Retrieval) и как обеспечивать надежность (Reliability) через встроенные механизмы Fallback и Retry.

**Стек:**
- **Inference:** Ollama + `qwen2.5-coder:3b`
- **Orchestration:** LangChain Expression Language (LCEL)
- **Validation:** Pydantic v2

# Настройка Среды

### 0. Настройка инфраструктуры
Поднимаем локальный сервер Ollama и скачиваем модель Qwen 2.5. Убедитесь, что в Colab включен GPU (T4).

In [53]:
# 1. Устанавливаем системную зависимость (zstd) для установки Ollama
!sudo apt-get update && sudo apt-get install -y zstd

# 2. Устанавливаем ядро стека 2026 года (LangGraph исключен)
!pip install -qU langchain-core langchain-ollama langchain-community chromadb sentence-transformers pydantic

# 3. Устанавливаем движок Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# 4. Запускаем сервер в фоне и качаем модель
import os
import subprocess
import time

print("Инициализация сервера Ollama...")
os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
with open("ollama.log", "w") as f:
    subprocess.Popen(["ollama", "serve"], stdout=f, stderr=f, env=os.environ)

# Даем серверу чуть больше времени на старт
time.sleep(7)

print("Скачивание Qwen 2.5 Coder (3B)...")
!ollama pull qwen2.5-coder:3b
print("✅ Окружение готово к Enterprise-разработке!")

Hit:1 https://cli.github.com/packages stable InRelease
Hit:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Hit:3 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:4 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Hit:10 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Hit:11 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Get:12 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,120 kB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,613 kB]
Fetched 5,861 kB in 3s (2,226 kB/s)
Reading packag

# Pydantic Guardrails

### 1. Schema Engineering: Строгая типизация и Pydantic Guardrails (Слайды 6, 10)

В 2026 году **The Schema is the Prompt**. Мы не просим модель текстово "выведи JSON в таком-то формате". Мы передаем ей строгую Pydantic-схему через `with_structured_output`.
Особое внимание обращаем на типы `Literal` и `Field` — это бесплатный способ защититься от галлюцинаций аргументов (Argument Hallucinations).

In [54]:
from pydantic import BaseModel, Field, ValidationError
from typing import Literal
from langchain_ollama import ChatOllama

# Инициализируем модель с нулевой температурой
llm = ChatOllama(model="qwen2.5-coder:3b", temperature=0.0)

# 1. Проектируем строгую схему. Literal запрещает модели выдумывать свои статусы.
class DatabaseQuery(BaseModel):
    """Схема для безопасного запроса метрик из базы данных."""
    metric_name: Literal["cpu_usage", "ram_usage", "disk_io"] = Field(
        ...,
        description="Название метрики. Выбери строго из доступных вариантов."
    )
    server_region: Literal["EU", "US", "RU"] = Field(
        ...,
        description="Регион сервера."
    )
    limit: int = Field(
        default=10,
        description="Количество записей. Не более 100.",
        le=100 # Pydantic Guardrail: limit must be <= 100
    )

# Привязываем схему к модели
structured_llm = llm.with_structured_output(DatabaseQuery)

### 📝 Задание 1: LCEL Цикл самокоррекции (Feedback Loop)
В реальной жизни модель может проигнорировать `Literal` или передать строку вместо числа.
Напишите функцию на чистом Python, которая делает запрос к `structured_llm`. Если возникает `ValidationError`, функция ловит её (except ValidationError) и отправляет текст ошибки обратно модели с требованием исправить JSON.

In [55]:
def generate_with_correction(user_query: str, max_retries: int = 3):
    messages =[
        ("system", "Ты DevOps-ассистент. Твоя задача извлечь параметры для SQL-запроса."),
        ("user", user_query)
    ]

    # TODO: Напишите цикл (for/while) для обработки Exception
    # В случае ошибки добавляйте в список messages два сообщения:
    # 1. ("assistant", "Invalid JSON")
    # 2. ("user", f"Ошибка валидации: {e}. Исправь данные под схему.")

    for i in range(max_retries):
        print(f"попытка {i}")
        try:
            # Запрашиваем структурированный ответ
            result = structured_llm.invoke(messages)
            print(f"парсинг выполнен")
            return result  # Успех — возвращаем валидный объект DatabaseQuery

        except ValidationError as e:
            # Добавляем два сообщения, как требуется
            messages.append(("assistant", "Invalid JSON"))
            messages.append(("user", f"Ошибка валидации: {e}. Исправь данные под схему."))
            print(f"ошибка валидации {str(e)[:150]}")

    raise RuntimeError("Модель не смогла выдать корректный структурированный ответ после всех попыток.")

# ТЕСТ (Модель должна понять, что 'ОАЭ' и 'network' - невалидны, и попытаться адаптировать или использовать дефолты)
print(generate_with_correction("Дай мне логи сети для серверов в ОАЭ, 500 строчек!"))

попытка 0
парсинг выполнен
metric_name='cpu_usage' server_region='US' limit=50


<details>
<summary><b>💡 Показать решение (Задание 1)</b></summary>

```python
def generate_with_correction(user_query: str, max_retries: int = 3):
    messages =[
        ("system", "Ты DevOps-ассистент. Извлеки параметры. Если регион или метрика не совпадают с Literal, подбери ближайший или верни EU / cpu_usage."),
        ("user", user_query)
    ]
    
    for attempt in range(max_retries):
        try:
            print(f"Попытка {attempt + 1}...")
            # Пытаемся получить структурированный ответ
            result = structured_llm.invoke(messages)
            print("✅ Успешно распарсено!")
            return result
        except Exception as e: # Ловим ошибки парсинга Pydantic
            error_msg = str(e)
            print(f"❌ Ошибка валидации:\n{error_msg[:150]}...")
            
            # Сохраняем историю ошибки для модели (Feedback Loop)
            messages.append(("assistant", "I generated an invalid output."))
            messages.append(("user", f"Твой ответ вызвал ошибку валидации Pydantic: {error_msg}. Исправь значения согласно схеме."))
            
    print("🚨 Достигнут лимит ретраев. Fallback на дефолтные значения.")
    return DatabaseQuery(metric_name="cpu_usage", server_region="EU", limit=10)

# Запуск
print(generate_with_correction("Дай мне логи сети (network) для серверов в ОАЭ, 500 строчек!"))
```
</details>

#LCEL Semantic Router

### 2. Маршрутизация интентов (Semantic Router) (Слайд 11)

Вместо того чтобы гнать все запросы через тяжелые RAG-цепочки, мы ставим дешевый классификатор на входе. В LCEL это элегантно решается с помощью `RunnableBranch` или кастомной `RunnableLambda`.

In [56]:
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

# 1. Быстрый классификатор намерений
class IntentRouter(BaseModel):
    intent: Literal["RAG", "ACTION", "CHITCHAT"] = Field(
        description="RAG: поиск информации в доках. ACTION: выполнение команд (перезагрузка, деплой). CHITCHAT: болтовня."
    )

router_llm = llm.with_structured_output(IntentRouter)

def extract_intent(state: dict) -> dict:
    query = state["query"]
    try:
        decision = router_llm.invoke(query)
        intent = decision.intent
    except:
        intent = "CHITCHAT" # Fallback
    print(f"[ROUTER] Намерение: {intent}")
    return {"intent": intent, "query": query}

# 2. Создаем три разные цепочки (Chains) для разных задач
rag_chain = (
    ChatPromptTemplate.from_template("📚 [RAG] Ищу в базе политик ответ на: {query}")
    | llm | StrOutputParser()
)

action_chain = (
    ChatPromptTemplate.from_template("🛠️ [ACTION] Подготавливаю bash-скрипт для: {query}")
    | llm | StrOutputParser()
)

chitchat_chain = (
    ChatPromptTemplate.from_template("💬 [CHITCHAT] Ответь коротко и вежливо на: {query}")
    | llm | StrOutputParser()
)

### 📝 Задание 2: Сборка LCEL-цепочки с маршрутизацией
Вам нужно собрать единую LCEL цепочку (enterprise_router_chain).
Она должна начинаться с `RunnableLambda(extract_intent)`, а затем использовать паттерн `if/elif` внутри другой `RunnableLambda` для вызова правильной цепочки (`rag_chain`, `action_chain` или `chitchat_chain`).

In [60]:
# TODO: Реализуйте функцию маршрутизации (route_to_chain) и соберите пайплайн.
def route_to_chain(state: dict):
    intent = state.get("intent", "CHITCHAT")

    if intent == "RAG":
        return rag_chain
    elif intent == "ACTION":
        return action_chain
    else:
        return chitchat_chain

enterprise_router_chain = (
    RunnablePassthrough.assign(intent=extract_intent)
    | RunnableLambda(route_to_chain)
)
print("--- Тест 1 ---")
print(enterprise_router_chain.invoke({"query": "Как оформить отпуск по регламенту?"}))
print("\n--- Тест 2 ---")
print(enterprise_router_chain.invoke({"query": "Перезагрузи production сервер БД"}))
print("\n--- Тест 3 ---")
print(enterprise_router_chain.invoke({"query": "Как твои дела, железяка?"}))

--- Тест 1 ---
[ROUTER] Намерение: CHITCHAT
Для оформления отпуска по регламенту необходимо обратиться в соответствующий отдел или центр вашего предприятия. Обычно это включает в себя заполнение специальной формы, указание даты начала и окончания отпуска, а также предоставление необходимых документов, таких как справка о доходах или справка о занятости. После этого форма отправляется на проверку руководителя или специалиста по управлению персоналом.

--- Тест 2 ---
[ROUTER] Намерение: ACTION
Конечно, перезагрузка сервера базы данных (production) может потребовать специальных прав и знаний. Рекомендуется обратиться к специалисту или администратору базы данных для выполнения этой операции.

--- Тест 3 ---
[ROUTER] Намерение: ACTION
Привет! Я здесь, чтобы помочь. Как я могу вам помочь сегодня?


<details>
<summary><b>💡 Показать решение (Задание 2)</b></summary>

```python
def route_to_chain(state: dict):
    intent = state["intent"]
    if intent == "RAG":
        return rag_chain
    elif intent == "ACTION":
        return action_chain
    else:
        return chitchat_chain

# Сборка финального LCEL пайплайна
# RunnableLambda(route_to_chain) вернет нужный Chain, и LCEL автоматически передаст в него state!
enterprise_router_chain = (
    RunnableLambda(extract_intent)
    | RunnableLambda(route_to_chain)
)

# Тесты
print("--- Тест 1 ---")
print(enterprise_router_chain.invoke({"query": "Как оформить отпуск по регламенту?"}))
print("\n--- Тест 2 ---")
print(enterprise_router_chain.invoke({"query": "Перезагрузи production сервер БД"}))
print("\n--- Тест 3 ---")
print(enterprise_router_chain.invoke({"query": "Как твои дела, железяка?"}))
```
</details>

# Dynamic Tool Retrieval
### 3. Tool Retrieval: Динамический контекст инструментов (Слайд 7)

Если у вас 100+ инструментов (Jira, Confluence, AWS, Git), вы не можете передать их все в `llm.bind_tools()`. Контекст переполнится, и модель начнет путать аргументы.
Решение — хранить описания инструментов в векторной БД и "подтягивать" только релевантные под запрос пользователя!

In [ ]:
from langchain_core.tools import tool
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. Создаем "Зоопарк" инструментов
@tool
def reset_password(user_id: str):
    """Сбрасывает пароль пользователя в Active Directory."""
    return f"Password reset for {user_id}"

@tool
def get_jira_ticket(ticket_id: str):
    """Получает статус задачи из Jira."""
    return f"Status of {ticket_id} is IN PROGRESS"

@tool
def restart_pod(pod_name: str):
    """Перезапускает pod в Kubernetes кластере."""
    return f"Pod {pod_name} restarted"

@tool
def calculate_bonus(employee_id: str):
    """Считает годовой бонус сотрудника."""
    return f"Bonus for {employee_id} is 5000"

all_tools =[reset_password, get_jira_ticket, restart_pod, calculate_bonus]

# 2. Индексируем инструменты в Векторную БД
# Мы векторизуем Docstrings инструментов
tool_docs =[t.description for t in all_tools]
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

vectorstore = Chroma.from_texts(
    texts=tool_docs,
    embedding=embeddings,
    metadatas=[{"tool_name": t.name} for t in all_tools]
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 1}) # Берем только 1 самый подходящий инструмент!

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


### 📝 Задание 3: RAG для инструментов
Напишите функцию `execute_with_dynamic_tools(query: str)`.
1. Сделайте векторный поиск по `query` в `retriever`.
2. Извлеките имя найденного инструмента из `metadata`.
3. Найдите этот инструмент в списке `all_tools`.
4. Сделайте `llm.bind_tools([найденный_инструмент])` и вызовите модель с `query`.
5. Выведите, какой инструмент модель решила вызвать.

In [61]:
def execute_with_dynamic_tools(query: str):
    print(f"\n[USER QUERY]: {query}")
    system_prompt = """
    Ты — ассистент, который ДОЛЖЕН использовать инструмент, если он подходит.
    Если инструмент подходит — обязательно вызывай его через tool call.
    Не отвечай текстом, если можно вызвать инструмент.
    """

    # 1. Находим лучший инструмент
    docs = retriever.invoke(query)
    tool_name = docs[0].metadata["tool_name"]
    print(f"[VECTOR SEARCH]: выбран инструмент -> {tool_name}")

    # 2. Достаём объект инструмента
    active_tool = next(t for t in all_tools if t.name == tool_name)

    # 3. Привязываем только его
    llm_with_tool = llm.bind_tools([active_tool])
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": query},
    ]

    # 4. Запускаем LLM
    response = llm_with_tool.invoke(messages)

    # 5. Обрабатываем Tool Call
    if response.tool_calls:
        for call in response.tool_calls:
            print(f"[LLM DECISION]: вызывает {call['name']} с аргументами {call['args']}")
            tool_obj = next(t for t in all_tools if t.name == call["name"])
            result = tool_obj.run(call["args"])
            print(f"[TOOL RESULT]: {result}")
    else:
        print("[LLM DECISION]: текстовый ответ:", response.content)


# -----------------------------
# 5. Тесты
# -----------------------------
execute_with_dynamic_tools("поменяй пароль пользователю c id =42")
execute_with_dynamic_tools("подними упавший pod в k8s")
execute_with_dynamic_tools("какой статус задачи с id = 234?")


[USER QUERY]: поменяй пароль пользователю c id =42
[VECTOR SEARCH]: выбран инструмент -> calculate_bonus
[LLM DECISION]: текстовый ответ: {"name": "change_password", "arguments": {"user_id": "42"}}

[USER QUERY]: подними упавший pod в k8s
[VECTOR SEARCH]: выбран инструмент -> restart_pod
[LLM DECISION]: текстовый ответ: {"name": "restart_pod", "arguments": {"pod_name": "your_pod_name"}}

[USER QUERY]: какой статус задачи с id = 234?
[VECTOR SEARCH]: выбран инструмент -> get_jira_ticket
[LLM DECISION]: текстовый ответ: {"name": "get_jira_ticket", "arguments": {"ticket_id": "234"}}


<details>
<summary><b>💡 Показать решение (Задание 3)</b></summary>

```python
def execute_with_dynamic_tools(query: str):
    print(f"\n[USER QUERY]: {query}")
    
    # 1. Ищем релевантный инструмент в векторной БД
    docs = retriever.invoke(query)
    found_tool_name = docs[0].metadata["tool_name"]
    print(f"[VECTOR SEARCH]: Найден подходящий инструмент -> {found_tool_name}")
    
    # 2. Достаем сам объект инструмента из списка
    active_tool = next(t for t in all_tools if t.name == found_tool_name)
    
    # 3. Привязываем ТОЛЬКО этот инструмент к модели (экономим контекст!)
    llm_with_tool = llm.bind_tools([active_tool])
    
    # 4. Вызываем модель
    response = llm_with_tool.invoke(query)
    
    # 5. Проверяем Tool Calls
    if response.tool_calls:
        for tc in response.tool_calls:
            print(f"[LLM DECISION]: Модель вызывает {tc['name']} с аргументами {tc['args']}")
    else:
        print("[LLM DECISION]: Модель ответила текстом:", response.content)

# Тесты
execute_with_dynamic_tools("Пользователь забыл пароль, его логин ivanov_i. Помоги.")
execute_with_dynamic_tools("Упал микросервис auth-service в k8s, подними его.")
```
</details>

# Reliability Engineering

### 4. Обработка ошибок в Enterprise (Fallbacks & Retries) (Слайд 12)

В продакшене API отваливаются, а модели выдают 503 ошибки. В LCEL есть мощнейшие механизмы `.with_retry()` и `.with_fallbacks()`, которые позволяют строить каскадные системы (например, переключаться с падающей Qwen на резервную Llama).

In [63]:
from langchain_core.runnables import RunnableLambda

# Симулируем нестабильную облачную модель (которая падает в 100% случаев)
def unstable_llm_call(prompt: str):
    print("🌐 [Cloud LLM]: Пытаюсь сгенерировать ответ...")
    raise Exception("503 Service Unavailable (Cloud is down!)")

unstable_chain = RunnableLambda(unstable_llm_call)

# Надежная локальная модель
reliable_local_chain = (
    ChatPromptTemplate.from_template("Ответь на запрос: {query}")
    | llm
    | StrOutputParser()
)

### 📝 Задание 4: Каскадный Fallback
Соберите `enterprise_resilient_chain`.
1. Она должна попытаться выполнить `unstable_chain`.
2. Если `unstable_chain` падает, она должна автоматически использовать механизм `.with_fallbacks([reliable_local_chain])`.

In [64]:
# TODO: Примените .with_fallbacks
# enterprise_resilient_chain = ...

# print(enterprise_resilient_chain.invoke({"query": "Какая столица Франции?"}))
unstable_with_retry = unstable_chain.with_retry(
    stop_after_attempt=3
)
enterprise_resilient_chain = unstable_with_retry.with_fallbacks([reliable_local_chain])

print("Запуск надежного пайплайна...")
result = enterprise_resilient_chain.invoke({"query": "Какая столица Франции?"})

print("\n[ФИНАЛЬНЫЙ ОТВЕТ]:")
print(result)


Запуск надежного пайплайна...
🌐 [Cloud LLM]: Пытаюсь сгенерировать ответ...
🌐 [Cloud LLM]: Пытаюсь сгенерировать ответ...
🌐 [Cloud LLM]: Пытаюсь сгенерировать ответ...

[ФИНАЛЬНЫЙ ОТВЕТ]:
Столица Франции - Париж.


<details>
<summary><b>💡 Показать решение (Задание 4)</b></summary>

```python
# Настраиваем каскадный Fallback
# Если unstable_chain падает, LCEL автоматически перехватит ошибку
# и прозрачно для пользователя направит запрос в reliable_local_chain
enterprise_resilient_chain = unstable_chain.with_fallbacks([reliable_local_chain])

print("Запуск надежного пайплайна...")
result = enterprise_resilient_chain.invoke({"query": "Какая столица Франции?"})

print("\n[ФИНАЛЬНЫЙ ОТВЕТ]:")
print(result)

# P.S. В реальных проектах также используется .with_retry(stop_after_attempt=3)
# для повторных попыток перед переходом к fallback-модели.
```
</details>



### 🎉 Итоги Модуля 3
Вы освоили фундаментальные Enterprise-паттерны LangChain (без использования графов):
1. **Pydantic Guardrails**: Использование `Literal` и циклов самокоррекции для защиты от галлюцинаций.
2. **Semantic Routing**: Маршрутизация запросов для экономии ресурсов GPU через `RunnableLambda`.
3. **Tool Retrieval**: Векторный поиск инструментов (RAG для инструментов) для преодоления лимитов контекста.
4. **Reliability**: Использование `.with_fallbacks()` для обеспечения отказоустойчивости (High Availability) систем.

В следующем модуле мы перейдем к LangGraph, чтобы добавить этим цепочкам долговременную память (Persistence) и паттерны Human-in-the-Loop!